# NumCompute-Stream Demo

This notebook shows the full streaming pipeline end to end.
We load a CSV dataset, split it into chunks to simulate a data stream,
train three models incrementally using partial_fit, then compare their
accuracy over time using the visualise module.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from numcompute_stream.io import read_csv, make_stream_chunks
from numcompute_stream.preprocessing import StandardScaler, Imputer
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.ensemble import RandomForestClassifier, BaggingClassifier
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.metrics import AccuracyTracker, ConfusionMatrixTracker
import numcompute_stream.visualise as vis

print('imports ok')

## 1. Load dataset from CSV

In [ ]:
headers, data = read_csv('penguins.csv')
print('headers:', headers)
print('shape:', data.shape)

# last column is the label, rest are features
X_all = data[:, :-1]
y_all = data[:, -1].astype(int)
print('classes:', np.unique(y_all))
print('class counts:', np.bincount(y_all))

## 2. Split into chunks to simulate streaming

In [ ]:
CHUNK_SIZE = 10
chunks = make_stream_chunks(X_all, y_all, chunk_size=CHUNK_SIZE)
print(f'{len(chunks)} chunks of up to {CHUNK_SIZE} samples each')
for i, (xc, yc) in enumerate(chunks):
    print(f'  chunk {i+1}: {xc.shape[0]} samples, classes {np.unique(yc)}')

## 3. Set up three pipelines for comparison

In [ ]:
def make_pipeline(model):
    return Pipeline([
        ('impute', Imputer(strategy='mean')),
        ('scale', StandardScaler()),
        ('model', model),
    ])

pipe_tree = make_pipeline(DecisionTreeClassifier(max_depth=4, random_state=0))
pipe_rf   = make_pipeline(RandomForestClassifier(n_estimators=10, max_depth=4, random_state=0))
pipe_bag  = make_pipeline(BaggingClassifier(n_estimators=10, max_depth=4, random_state=0))

trainer_tree = StreamTrainer(pipe_tree, classes=[0, 1, 2])
trainer_rf   = StreamTrainer(pipe_rf,   classes=[0, 1, 2])
trainer_bag  = StreamTrainer(pipe_bag,  classes=[0, 1, 2])

print('pipelines ready')

## 4. Train incrementally chunk by chunk

In [ ]:
for X_chunk, y_chunk in chunks:
    trainer_tree.fit_chunk(X_chunk, y_chunk)
    trainer_rf.fit_chunk(X_chunk, y_chunk)
    trainer_bag.fit_chunk(X_chunk, y_chunk)

print('training complete')
print()
print('--- Decision Tree ---')
trainer_tree.summary()
print()
print('--- Random Forest ---')
trainer_rf.summary()

## 5. Plot accuracy over time for each model

In [ ]:
vis.plot_metric_over_time(
    trainer_tree.cumulative_accuracy_history(),
    title='Decision Tree: cumulative accuracy over chunks',
    ylabel='accuracy'
)

vis.plot_metric_over_time(
    trainer_rf.cumulative_accuracy_history(),
    title='Random Forest: cumulative accuracy over chunks',
    ylabel='accuracy',
    color='seagreen'
)

## 6. Compare Random Forest vs Bagging

In [ ]:
vis.compare_models(
    trainer_rf.cumulative_accuracy_history(),
    trainer_bag.cumulative_accuracy_history(),
    labels=('Random Forest', 'Bagging'),
    title='RF vs Bagging: cumulative accuracy',
    ylabel='accuracy'
)

## 7. Predictions vs ground truth on the last chunk

In [ ]:
X_last, y_last = chunks[-1]
preds_rf = pipe_rf.predict(X_last)

vis.plot_predictions_vs_ground_truth(
    y_last,
    preds_rf,
    title='Random Forest: predictions vs ground truth (last chunk)'
)

## 8. Confusion matrix on full dataset

In [ ]:
preds_all = pipe_rf.predict(X_all)
cmt = ConfusionMatrixTracker()
cmt.update(y_all, preds_all)
cm_result = cmt.result()

vis.plot_confusion_matrix(
    cm_result['matrix'],
    classes=['Adelie', 'Chinstrap', 'Gentoo'],
    title='Random Forest confusion matrix (full dataset)'
)

## 9. Training time per chunk

In [ ]:
elapsed = [e['elapsed_s'] for e in trainer_rf.chunk_log]
vis.plot_chunk_times(elapsed, title='Random Forest: training time per chunk')

## 10. Loss over time

In [ ]:
vis.plot_metric_over_time(
    trainer_rf.loss_history(),
    title='Random Forest: loss over chunks',
    ylabel='MSE loss',
    color='tomato'
)